In this worksheet, we are trying to break up the tables in data cleaning into a suitable table for modelling. Right now, the cleaned data has multiple columns in one row of various AC load, which means per row, we have more than one y variable. Hence, we need to break them up, such that each row only has one y variable which explains parts of what this file is doing

`[Cell 1]`

# Question B: modelling dataset

**Question B:** Can we accurately forecast a zone's AC energy demand by combining historical indoor environmental conditions with external weather forecasts?

This notebook turns the cleaned hourly data into the **modelling dataset** shared by the baselines, LightGBM and XGBoost: **one row per zone per hour**, with the target, the features and the train / validation / test labels. It also writes the **feature dictionary**.

**Run first:** `01_clean_raw_to_hourly.ipynb` (raw Kaggle files → `question_b_hourly_data.parquet` and `question_b_minute_data.parquet`).

**Key decisions**

| Decision | Choice |
|---|---|
| Forecast | **Day-ahead**: made at midnight for the next day's 24 hours. Every feature must be known by then |
| Target | `ac_load_kw`: the zone's AC demand, all its working units added up, averaged over the hour (kW = kWh for that hour) |
| Lags | 24 hours or more only (yesterday, 2 days ago, last week). The previous hour (t-1) is **not** included: it is not known a day ahead |
| Weather | Observed weather (Open-Meteo) as a stand-in for weather forecasts |
| Hours | All 24 hours kept; `is_operating_hours` marks weekday working hours so accuracy can also be reported for them alone |
| Missing values | Not filled. A zone-hour with any unit missing has no target; those rows are **removed** from the saved dataset, so every row has an AC load. Missing **features** (lags) are left as `NaN`: LightGBM and XGBoost handle them |
| Splits | **Train:** 2019 except April, August and December · **Validation:** April (hot season), August (rainy season) and December (cool season) 2019 · **Test:** July–September 2018 (a different year, used once at the end) |

**Outputs:** `question_b_model_data.parquet` (and `.csv`): the modelling dataset, only rows with a known AC load. `question_b_feature_dictionary.csv`. `question_b_model_data_full.parquet`: a backup with every zone and every hour, including the hours with no AC load (for building new lag features later).

In [1]:
# [Cell 2]
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)

`[Cell 3]`

## 1. Load the cleaned hourly data

In [2]:
# [Cell 4]
hourly = pd.read_parquet('question_b_hourly_data.parquet')       # one row per floor per hour
print(hourly.shape, '| from', hourly['Hour'].min(), 'to', hourly['Hour'].max())

(55622, 52) | from 2018-07-01 00:00:00 to 2019-12-31 23:00:00


`[Cell 5]`

## 2. The 23 zones and their AC units

A zone is used if it has at least one working AC unit. A unit whose reading never changes (a dead meter: `z2_AC11` on Floor 2, `z1_AC1` on Floor 7) is left out. Zone 3 has no AC.

In [3]:
# [Cell 6]
ac_cols = [c for c in hourly.columns if '_AC' in c]

zone_units = {}
for floor, g in hourly.groupby('Floor'):
    for zone in [1, 2, 4, 5]:
        units = [c for c in ac_cols
                 if c.startswith(f'z{zone}_AC') and g[c].notna().any() and g[c].std() > 0]
        # the above line of code would exclude z2_AC11 and z1_AC1 because both ac has a min and max ac load of 0, resulting in standard deviation of 0
        if units:
            zone_units[(floor, zone)] = units

print(len(zone_units), 'zones')
pd.DataFrame([{'zone_id': f'F{f}_Z{z}', 'AC units': len(u), 'units': ', '.join(u)}
              for (f, z), u in zone_units.items()])

23 zones


,zone_id,AC units,units
0,F2_Z1,1,z1_AC1(kW)
1,F2_Z2,13,"z2_AC1(kW), z2_AC2(kW), z2_AC3(kW), z2_AC4(kW)..."
2,F2_Z4,1,z4_AC1(kW)
3,F3_Z1,4,"z1_AC1(kW), z1_AC2(kW), z1_AC3(kW), z1_AC4(kW)"
4,F3_Z2,1,z2_AC1(kW)
5,F3_Z4,1,z4_AC1(kW)
6,F3_Z5,1,z5_AC1(kW)
7,F4_Z1,4,"z1_AC1(kW), z1_AC2(kW), z1_AC3(kW), z1_AC4(kW)"
8,F4_Z2,1,z2_AC1(kW)
9,F4_Z4,1,z4_AC1(kW)


`[Cell 7]`

## 3. Building-wide hourly values: weather and holidays

Weather and holidays are the same for every zone, so they are taken once (from Floor 2, whose record covers every hour of the two periods) and renamed to the modelling names.

The cell below basically 

1.) Renames the outdoor weather column into something more details <br>
2.) Covert holiday flag from yes and no to boolean

In [4]:
# [Cell 8]
weather_names = {
    'hourly outdoor temp (degC)':    'outdoor_temp_hourly_c',
    'outdoor temp (degC)':           'outdoor_temp_daily_avg_c',
    'hourly outdoor humidity (%)':   'outdoor_humidity_hourly_pct',
    'outdoor humidity (%)':          'outdoor_humidity_daily_avg_pct',
    'hourly rainfall (mm)':          'rainfall_hourly_mm',
    'rainfall (mm)':                 'rainfall_daily_total_mm',
    'hourly solar radiation (W/m2)': 'solar_radiation_hourly_wm2',
    'solar radiation (MJ/m2)':       'solar_radiation_daily_total_mjm2',
}
building = (hourly[hourly['Floor'] == 2].set_index('Hour')[list(weather_names) + ['holiday?']]
            .rename(columns=weather_names))
building['is_holiday'] = (building.pop('holiday?') == 'Yes').astype(int)
print(building.shape, '| missing weather values:', building[list(weather_names.values())].isna().sum().sum())

(9552, 9) | missing weather values: 0


`[Cell 9]`

## 4. Each zone's own indoor sensor: out-of-range readings

The hourly data flags a floor-hour if **any** zone's sensor was out of range. For the zone-level dataset the check is redone on each zone's **own** sensor, from the minute data: 'out of range' if any minute in the hour had indoor temperature outside 15–36 °C, or humidity outside 20–100%.

In [5]:
# [Cell 10]
s1_cols = [f'z{z}_S1({u})' for z in [1, 2, 4, 5] for u in ['degC', 'RH%']]
minute = pd.read_parquet('question_b_minute_data.parquet', columns=['Floor', 'Date'] + s1_cols)
minute['Hour'] = minute['Date'].dt.floor('h')

out_of_range = {}                        # (floor, zone) -> hourly 0/1 flags
for floor, g in minute.groupby('Floor'):
    for zone in [1, 2, 4, 5]:
        temp, rh = g[f'z{zone}_S1(degC)'], g[f'z{zone}_S1(RH%)']
        flags = pd.DataFrame({
            'indoor_temp_out_of_range_flag': temp.notna() & ~temp.between(15, 36),
            'indoor_humidity_out_of_range_flag': rh.notna() & ~rh.between(20, 100),
        }).groupby(g['Hour']).any().astype(int)
        out_of_range[(floor, zone)] = flags
del minute
print('zones checked:', len(out_of_range))

zones checked: 24


`[Cell 11]`

## 5. One row per zone per hour, with the lag features

For each zone, on a **continuous** hourly timeline (July 2018 – December 2019), so that "yesterday" and "last week" always point to the right hour. Hours that are not in the data (October 2018 – February 2019, and Floor 6 after 22 October 2019) are empty, so a lag that points into them is missing (`NaN`).

- `ac_load_kw`: the sum of the zone's units, **missing if any unit is missing** that hour (so the total is never undercounted).
- AC lags: `ac_load_kw_yesterday` (24 hours earlier), `ac_load_kw_2_days_ago` (48 hours), `ac_load_kw_last_week` (168 hours).
- Indoor lags: the same hour yesterday and last week, and **yesterday's average** (the mean of the previous calendar day, kept only if at least 22 of its 24 hours have a reading, the same 90% rule as the hourly averages).
- `ac_spike_flag`: 1 if any of **this zone's** units spiked during the hour.

The indoor readings of the same hour are only used to build the lags: they are not known a day ahead, so they are not kept as features.

In [6]:
# [Cell 12]
timeline = pd.date_range('2018-07-01 00:00', '2019-12-31 23:00', freq='h', name='timestamp')
LAGS = {'yesterday': 24, '2_days_ago': 48, 'last_week': 168}

zone_tables = []
for (floor, zone), units in zone_units.items():
    g = hourly[hourly['Floor'] == floor].set_index('Hour').reindex(timeline)
    t = pd.DataFrame(index=timeline)
    t['zone_id'], t['floor'], t['zone'] = f'F{floor}_Z{zone}', floor, zone

    # target
    t['ac_load_kw'] = g[units].sum(axis=1, min_count=len(units))

    # AC lags
    for name, hours in LAGS.items():
        t[f'ac_load_kw_{name}'] = t['ac_load_kw'].shift(hours)

    # indoor lags (the same-hour readings are not kept)
    for reading, unit_name in [(f'z{zone}_S1(degC)', 'indoor_temp_c'), (f'z{zone}_S1(RH%)', 'indoor_humidity_pct')]:
        now = g[reading]
        t[f'{unit_name}_yesterday'] = now.shift(24)
        t[f'{unit_name}_last_week'] = now.shift(168)
        day_mean = now.groupby(now.index.normalize()).mean().where(now.groupby(now.index.normalize()).count() >= 22)
        t[f'{unit_name}_yesterday_avg'] = (t.index.normalize() - pd.Timedelta(days=1)).map(day_mean)

    # flags for this zone
    unit_pattern = '|'.join(u.replace('(', r'\(').replace(')', r'\)') for u in units)
    t['ac_spike_flag'] = g['AC spike units'].fillna('').str.contains(unit_pattern).astype(int)
    flags = out_of_range[(floor, zone)].reindex(timeline, fill_value=False).astype(int)
    t['indoor_temp_out_of_range_flag'] = flags['indoor_temp_out_of_range_flag']
    t['indoor_humidity_out_of_range_flag'] = flags['indoor_humidity_out_of_range_flag']
    t['supply_air_sensor_flag'] = int((floor, zone) == (5, 1))
    zone_tables.append(t)

data = pd.concat(zone_tables).reset_index()
print(data.shape)

(303048, 18)


`[Cell 13]`

## 6. Keep the two periods, add the calendar, weather and split labels

- Keep July–September 2018 and March–December 2019 (the periods with indoor sensor readings).
- **Calendar:** `hour` (0–23), `weekday` (0 = Monday … 6 = Sunday), `month`, `is_weekend`, `is_holiday`, `day_type` (weekday / weekend / holiday) and `is_operating_hours` (1 on weekdays that are not holidays, 07:00–17:00, when the AC runs).
- **Zone encoding:** `zone_code`, a number from 0 to 22 for each zone (for the models), with `floor` and `zone`.
- **Split:** one validation month from each season of 2019, so that every season is also in the training data: April (hot), August (rainy) and December (cool). The rest of 2019 is training. July–September 2018 is the test: a different year, used once at the end.
- `has_target`: 1 if the zone's AC load is known for that hour.

In [7]:
# [Cell 14]
ts = data['timestamp']
in_periods = ts.between('2018-07-01', '2018-09-30 23:00') | ts.between('2019-03-01', '2019-12-31 23:00')
data = data[in_periods].reset_index(drop=True)
ts = data['timestamp']

data = data.join(building, on='timestamp')                      # weather + is_holiday

data['hour'] = ts.dt.hour
data['weekday'] = ts.dt.dayofweek
data['month'] = ts.dt.month
data['is_weekend'] = (data['weekday'] >= 5).astype(int)
data['day_type'] = np.select([data['is_holiday'] == 1, data['is_weekend'] == 1], ['holiday', 'weekend'], 'weekday')
data['is_operating_hours'] = ((data['day_type'] == 'weekday') & data['hour'].between(7, 16)).astype(int)
data['zone_code'] = data['zone_id'].map({z: i for i, z in enumerate(sorted(data['zone_id'].unique()))})

VALIDATION_MONTHS = [4, 8, 12]          # April (hot), August (rainy), December (cool), all 2019
data['split'] = np.select(
    [ts.dt.year == 2018, ts.dt.month.isin(VALIDATION_MONTHS)],
    ['test', 'validation'], 'train')
data['has_target'] = data['ac_load_kw'].notna().astype(int)

print(data.shape)
data.groupby('split')['timestamp'].agg(['min', 'max', 'count'])

(219696, 36)


,min,max,count
split,,,
test,2018-07-01,2018-09-30 23:00:00,50784
train,2019-03-01,2019-11-30 23:00:00,118128
validation,2019-04-01,2019-12-31 23:00:00,50784


`[Cell 15]`

## 7. Column order

In [8]:
# [Cell 16]
ids = ['zone_id', 'floor', 'zone', 'zone_code', 'timestamp', 'split', 'has_target']
target = ['ac_load_kw']
calendar = ['hour', 'weekday', 'month', 'is_weekend', 'is_holiday', 'day_type', 'is_operating_hours']
ac_lags = ['ac_load_kw_yesterday', 'ac_load_kw_2_days_ago', 'ac_load_kw_last_week']
indoor_lags = ['indoor_temp_c_yesterday', 'indoor_temp_c_last_week', 'indoor_temp_c_yesterday_avg',
               'indoor_humidity_pct_yesterday', 'indoor_humidity_pct_last_week', 'indoor_humidity_pct_yesterday_avg']
weather = list(weather_names.values())
flags = ['ac_spike_flag', 'indoor_temp_out_of_range_flag', 'indoor_humidity_out_of_range_flag', 'supply_air_sensor_flag']

data = data[ids + target + calendar + ac_lags + indoor_lags + weather + flags]
print(data.shape)
data[data['zone_id'] == 'F2_Z2'].set_index('timestamp').loc['2019-04-03 06:00':'2019-04-03 11:00']

(219696, 36)


,zone_id,floor,zone,zone_code,split,has_target,ac_load_kw,hour,weekday,month,is_weekend,is_holiday,day_type,is_operating_hours,ac_load_kw_yesterday,ac_load_kw_2_days_ago,ac_load_kw_last_week,indoor_temp_c_yesterday,indoor_temp_c_last_week,indoor_temp_c_yesterday_avg,indoor_humidity_pct_yesterday,indoor_humidity_pct_last_week,indoor_humidity_pct_yesterday_avg,outdoor_temp_hourly_c,outdoor_temp_daily_avg_c,outdoor_humidity_hourly_pct,outdoor_humidity_daily_avg_pct,rainfall_hourly_mm,rainfall_daily_total_mm,solar_radiation_hourly_wm2,solar_radiation_daily_total_mjm2,ac_spike_flag,indoor_temp_out_of_range_flag,indoor_humidity_out_of_range_flag,supply_air_sensor_flag
timestamp,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2019-04-03 06:00:00,F2_Z2,2,2,1,validation,1,6.462000,6,2,4,0,0,weekday,0,6.346000,6.591500,6.917167,27.181667,26.309667,24.343397,62.390833,65.446667,68.952175,26.4,29.4,86,73,0.0,2.1,27.0,19.52,0,0,0,0
2019-04-03 07:00:00,F2_Z2,2,2,1,validation,1,6.590667,7,2,4,0,0,weekday,1,6.511500,6.689500,8.282667,27.346167,26.268500,24.343397,63.286833,65.096833,68.952175,25.9,29.4,88,73,0.0,2.1,163.0,19.52,0,0,0,0
2019-04-03 08:00:00,F2_Z2,2,2,1,validation,1,32.031167,8,2,4,0,0,weekday,1,31.460500,33.126500,37.495000,22.460500,21.796500,24.343397,70.447500,70.760333,68.952175,28.0,29.4,78,73,0.0,2.1,324.0,19.52,0,0,0,0
2019-04-03 09:00:00,F2_Z2,2,2,1,validation,1,33.550000,9,2,4,0,0,weekday,1,34.315333,34.999833,37.731833,21.080333,20.157500,24.343397,76.709833,76.618333,68.952175,29.1,29.4,74,73,0.2,2.1,533.0,19.52,0,0,0,0
2019-04-03 10:00:00,F2_Z2,2,2,1,validation,1,34.778000,10,2,4,0,0,weekday,1,36.263333,36.470667,35.974000,NaN,19.983000,24.343397,NaN,79.754667,68.952175,30.6,29.4,67,73,0.3,2.1,630.0,19.52,0,0,0,0
2019-04-03 11:00:00,F2_Z2,2,2,1,validation,1,31.825667,11,2,4,0,0,weekday,1,32.986667,34.775667,33.886167,NaN,20.267000,24.343397,NaN,80.022667,68.952175,31.5,29.4,64,73,0.3,2.1,734.0,19.52,0,0,0,0


`[Cell 17]`

## 8. Checks

Check that we have the right shape for the table and we don't have any duplicates

In [9]:
# [Cell 18]
# One row per zone per hour, 23 zones, no duplicates
print('rows:', len(data), '=', data['zone_id'].nunique(), 'zones x', data['timestamp'].nunique(), 'hours')
print('duplicate zone-hour rows:', data.duplicated(['zone_id', 'timestamp']).sum())

rows: 219696 = 23 zones x 9552 hours
duplicate zone-hour rows: 0


In [10]:
# [Cell 19]
# Leakage check: every lag must equal the target that many hours earlier (spot check on one zone)
z = data[data['zone_id'] == 'F2_Z2'].set_index('timestamp')['ac_load_kw']
row = data[(data['zone_id'] == 'F2_Z2') & (data['timestamp'] == '2019-04-03 09:00')].iloc[0]
for name, hours in LAGS.items():
    earlier = row['timestamp'] - pd.Timedelta(hours=hours)
    print(f'ac_load_kw_{name}: {row[f"ac_load_kw_{name}"]:.3f} | target at {earlier}: {z.get(earlier, np.nan):.3f}')

ac_load_kw_yesterday: 34.315 | target at 2019-04-02 09:00:00: 34.315
ac_load_kw_2_days_ago: 35.000 | target at 2019-04-01 09:00:00: 35.000
ac_load_kw_last_week: 37.732 | target at 2019-03-27 09:00:00: 37.732


In [11]:
# [Cell 20]
# Rows with a known target, by split (Floor 6 has no data after 22 Oct 2019, so December validation has 19 zones)
summary = data.groupby('split').agg(rows=('has_target', 'size'), with_target=('has_target', 'sum'),
                                    zones_with_data=('zone_id', lambda s: s[data.loc[s.index, 'has_target'] == 1].nunique()))
summary['with_target (%)'] = (summary['with_target'] / summary['rows'] * 100).round(1)
summary

,rows,with_target,zones_with_data,with_target (%)
split,,,,
test,50784,50567,23,99.6
train,118128,111779,23,94.6
validation,50784,46773,23,92.1


In [12]:
# [Cell 21]
# Share of rows with a value (%) for each feature, by split (rows with a target only)
features = calendar + ac_lags + indoor_lags + weather
with_target = data[data['has_target'] == 1]
with_target.groupby('split')[features].apply(lambda g: g.notna().mean().mul(100).round(1)).T

split,test,train,validation
hour,100.0,100.0,100.0
weekday,100.0,100.0,100.0
month,100.0,100.0,100.0
is_weekend,100.0,100.0,100.0
is_holiday,100.0,100.0,100.0
day_type,100.0,100.0,100.0
is_operating_hours,100.0,100.0,100.0
ac_load_kw_yesterday,98.5,99.2,99.6
ac_load_kw_2_days_ago,97.4,98.6,99.4
ac_load_kw_last_week,91.9,96.0,98.5


In [13]:
# [Cell 22]
# Operating hours and flags
pd.DataFrame({
    'rows': [len(data)],
    'operating hours': [data['is_operating_hours'].sum()],
    'AC spike hours': [data['ac_spike_flag'].sum()],
    'indoor temp out of range': [data['indoor_temp_out_of_range_flag'].sum()],
    'indoor humidity out of range': [data['indoor_humidity_out_of_range_flag'].sum()],
    'supply-air sensor rows': [data['supply_air_sensor_flag'].sum()],
})

,rows,operating hours,AC spike hours,indoor temp out of range,indoor humidity out of range,supply-air sensor rows
0,219696,60260,217,1160,0,9552


`[Cell 23]`

## 9. Feature dictionary

Every column, what it means, its unit, and its role. The last four columns say which feature set each feature belongs to:

- **M1:** calendar + AC lags (+ zone encoding)
- **M2:** M1 + indoor temperature / humidity lags
- **M3:** M1 + outdoor weather
- **M4:** M1 + indoor and outdoor

Every feature is **day-ahead safe**: it is known at midnight before the day being forecast (the weather comes from a forecast in real use; here observed weather stands in for it).

In [14]:
# [Cell 24]
d = [
    # column, role, meaning, unit
    ('zone_id', 'id', 'Zone label, e.g. F2_Z2 = Floor 2, zone 2', ''),
    ('floor', 'feature', 'Floor number (2-7)', ''),
    ('zone', 'id', 'Zone number on the floor (1, 2, 4, 5)', ''),
    ('zone_code', 'feature', 'Zone as a number from 0 to 22 (for the models)', ''),
    ('timestamp', 'id', 'Start of the hour', ''),
    ('split', 'split', 'train (2019 except Apr, Aug, Dec), validation (Apr, Aug, Dec 2019), test (Jul-Sep 2018)', ''),
    ('ac_load_kw', 'target', "Zone AC demand: the zone's working AC units added up, averaged over the hour (= kWh for the hour). Missing if any unit is missing", 'kW'),
    ('hour', 'feature', 'Hour of the day', '0-23'),
    ('weekday', 'feature', 'Day of the week', '0 = Monday ... 6 = Sunday'),
    ('month', 'feature', 'Month', '1-12'),
    ('is_weekend', 'feature', '1 on Saturday and Sunday', '0/1'),
    ('is_holiday', 'feature', '1 on a public holiday when the office was closed or largely closed (28 days)', '0/1'),
    ('day_type', 'baseline', 'weekday / weekend / holiday (text; used by baseline B2, not a model feature)', 'text'),
    ('is_operating_hours', 'filter', '1 on weekdays that are not holidays, 07:00-17:00 (when the AC runs); for reporting accuracy on working hours', '0/1'),
    ('ac_load_kw_yesterday', 'feature', 'ac_load_kw at the same hour 1 day earlier (t-24)', 'kW'),
    ('ac_load_kw_2_days_ago', 'feature', 'ac_load_kw at the same hour 2 days earlier (t-48)', 'kW'),
    ('ac_load_kw_last_week', 'feature', 'ac_load_kw at the same hour 1 week earlier (t-168)', 'kW'),
    ('indoor_temp_c_yesterday', 'feature', "The zone's indoor temperature at the same hour 1 day earlier", 'degC'),
    ('indoor_temp_c_last_week', 'feature', "The zone's indoor temperature at the same hour 1 week earlier", 'degC'),
    ('indoor_temp_c_yesterday_avg', 'feature', "The zone's average indoor temperature over the previous day (needs 22 of 24 hours)", 'degC'),
    ('indoor_humidity_pct_yesterday', 'feature', "The zone's indoor humidity at the same hour 1 day earlier", '%'),
    ('indoor_humidity_pct_last_week', 'feature', "The zone's indoor humidity at the same hour 1 week earlier", '%'),
    ('indoor_humidity_pct_yesterday_avg', 'feature', "The zone's average indoor humidity over the previous day (needs 22 of 24 hours)", '%'),
    ('outdoor_temp_hourly_c', 'feature', 'Outdoor temperature at the start of the hour (Open-Meteo, Bangkok)', 'degC'),
    ('outdoor_temp_daily_avg_c', 'feature', "The day's average outdoor temperature", 'degC'),
    ('outdoor_humidity_hourly_pct', 'feature', 'Outdoor relative humidity at the start of the hour', '%'),
    ('outdoor_humidity_daily_avg_pct', 'feature', "The day's average outdoor relative humidity", '%'),
    ('rainfall_hourly_mm', 'feature', 'Rain that fell during the hour', 'mm'),
    ('rainfall_daily_total_mm', 'feature', "The day's total rainfall", 'mm'),
    ('solar_radiation_hourly_wm2', 'feature', 'Average solar radiation during the hour', 'W/m2'),
    ('solar_radiation_daily_total_mjm2', 'feature', "The day's total solar radiation", 'MJ/m2'),
    ('ac_spike_flag', 'flag', "1 if any of the zone's AC units read above 1.5 x its 99.9th percentile during the hour (values kept)", '0/1'),
    ('indoor_temp_out_of_range_flag', 'flag', "1 if the zone's indoor temperature was outside 15-36 degC in any minute of the hour", '0/1'),
    ('indoor_humidity_out_of_range_flag', 'flag', "1 if the zone's indoor humidity was outside 20-100% in any minute of the hour", '0/1'),
    ('supply_air_sensor_flag', 'flag', "1 for every Floor 5 zone 1 row: its temperature sensor reads AC supply air, not room air", '0/1'),
]
dictionary = pd.DataFrame(d, columns=['column', 'role', 'meaning', 'unit'])
saved_columns = [c for c in data.columns if c != 'has_target']      # has_target is not in the saved dataset
assert list(dictionary['column']) == saved_columns, 'dictionary and dataset columns differ'

m1 = ['floor', 'zone_code'] + [c for c in calendar if c not in ('is_operating_hours', 'day_type')] + ac_lags
sets = {'M1': m1, 'M2': m1 + indoor_lags, 'M3': m1 + weather, 'M4': m1 + indoor_lags + weather}
for name, cols in sets.items():
    dictionary[name] = dictionary['column'].isin(cols).map({True: 'Yes', False: ''})

dictionary.to_csv('question_b_feature_dictionary.csv', index=False)
print({name: len(cols) for name, cols in sets.items()})
dictionary

{'M1': 10, 'M2': 16, 'M3': 18, 'M4': 24}


,column,role,meaning,unit,M1,M2,M3,M4
0,zone_id,id,"Zone label, e.g. F2_Z2 = Floor 2, zone 2",,,,,
1,floor,feature,Floor number (2-7),,Yes,Yes,Yes,Yes
2,zone,id,"Zone number on the floor (1, 2, 4, 5)",,,,,
3,zone_code,feature,Zone as a number from 0 to 22 (for the models),,Yes,Yes,Yes,Yes
4,timestamp,id,Start of the hour,,,,,
5,split,split,"train (2019 except Apr, Aug, Dec), validation ...",,,,,
6,ac_load_kw,target,Zone AC demand: the zone's working AC units ad...,kW,,,,
7,hour,feature,Hour of the day,0-23,Yes,Yes,Yes,Yes
8,weekday,feature,Day of the week,0 = Monday ... 6 = Sunday,Yes,Yes,Yes,Yes
9,month,feature,Month,1-12,Yes,Yes,Yes,Yes


`[Cell 25]`

## 10. Save

- `question_b_model_data.parquet` (for Python) and `question_b_model_data.csv` (to open in Excel): **only the rows with a known AC load** (`has_target` = 1), so they can be used directly. The `has_target` column is dropped, since it would always be 1.
- `question_b_model_data_full.parquet`: a backup with **every zone and every hour**, including the hours with no AC load (`has_target` = 0). Use it only to build new lag features: "24 rows back" is "yesterday" only on this complete timeline.

**For the LightGBM and XGBoost models:** use `question_b_model_data.parquet` as it is (no filtering needed); pick the columns for M1–M4 from `question_b_feature_dictionary.csv`; report accuracy for all hours and for `is_operating_hours == 1`. Tune on `validation`; then **retrain the chosen model on all of 2019 (train + validation)** and evaluate it **once** on `test` (July–September 2018). `day_type` is text: use `is_weekend` / `is_holiday` in the models instead, or convert it.

In [15]:
# [Cell 26]
# backup: every zone and every hour, including hours with no AC load
data.to_parquet('question_b_model_data_full.parquet', index=False)

# the modelling dataset: only the rows with a known AC load
model_data = data[data['has_target'] == 1].drop(columns='has_target').reset_index(drop=True)
model_data.to_parquet('question_b_model_data.parquet', index=False)
model_data.to_csv('question_b_model_data.csv', index=False)

check = pd.read_parquet('question_b_model_data.parquet')
print('full timeline:', data.shape, '| modelling dataset:', model_data.shape,
      '| rows removed (no AC load):', len(data) - len(model_data))
print('saved and reloaded OK:', check.equals(model_data), '| missing AC load left:', check['ac_load_kw'].isna().sum())

full timeline: (219696, 36) | modelling dataset: (209119, 35) | rows removed (no AC load): 10577
saved and reloaded OK: True | missing AC load left: 0
